# 01: Raw Logs to Training Records

Download DSAT scraper CSV files from GitHub, pair consecutive bus observations to derive per-segment trip times, and export a clean CSV for XGBoost training.

In [ ]:
!pip install pandas numpy haversine requests pyproj

In [ ]:
import os, requests, pandas as pd

RAW_URL = "https://raw.githubusercontent.com/ChiHin-Lio/macau-gtfs-data/main/raw_logs"
ROUTES = ["25B", "MT4", "33", "26A", "3", "22"]

os.makedirs("raw_logs", exist_ok=True)
all_dfs = []
for route in ROUTES:
    url = f"{RAW_URL}/{route}.csv"
    try:
        df = pd.read_csv(url)
        df["route"] = route
        all_dfs.append(df)
        print(f"{route}: {len(df)} rows")
    except Exception as e:
        print(f"{route}: not found ({e})")

if not all_dfs:
    raise RuntimeError("No data. Run the scraper first!")

raw = pd.concat(all_dfs, ignore_index=True)
raw["timestamp"] = pd.to_datetime(raw["timestamp"])
raw = raw.sort_values(["busPlate", "timestamp"])
print(f"\nTotal raw records: {len(raw)}")
print(f"Date range: {raw['timestamp'].min()} to {raw['timestamp'].max()}")

In [ ]:
# Load stop coordinates from ParkGo bus-stops.json
STOPS_URL = "https://raw.githubusercontent.com/ChiHin-Lio/macau-gtfs-data/main/bus-stops.json"
stops_resp = requests.get(STOPS_URL)
stops_data = stops_resp.json()

stop_coords = {}
for s in stops_data:
    lat, lon = s["coordinates"][1], s["coordinates"][0]
    stop_coords[s["id"]] = (lat, lon)

print(f"Loaded {len(stop_coords)} stops")

In [ ]:
from haversine import haversine, Unit
import numpy as np

training_records = []

# Optional: also load DSAT official BUS_POLE.csv if available for better WGS84 coords
DSAT_STOPS_URL = "https://raw.githubusercontent.com/ChiHin-Lio/macau-gtfs-data/main/dsat-data/BUS_POLE.csv"
try:
    dsat_stops = pd.read_csv(DSAT_STOPS_URL)
    dsat_coords = {}
    for _, row in dsat_stops.iterrows():
        sid = str(row["POLE_ID"])
        alias = str(row["P_ALIAS"]).strip()
        if pd.notna(row["lng"]) and pd.notna(row["lat"]):
            dsat_coords[alias] = (row["lat"], row["lng"])
    # Merge into stop_coords (DSAT official takes priority)
    stop_coords.update(dsat_coords)
    print(f"Merged {len(dsat_coords)} DSAT official stop coordinates")
except Exception as e:
    print(f"DSAT BUS_POLE not available, using ParkGo stops only: {e}")

for bus, group in raw.groupby("busPlate"):
    group = group.sort_values("timestamp")
    prev_row = None
    prev_latlon = None
    for _, row in group.iterrows():
        if prev_row is None:
            prev_row = row
            prev_latlon = stop_coords.get(row["staCode"])
            continue
        if row["staCode"] == prev_row["staCode"]:
            continue
        cur_latlon = stop_coords.get(row["staCode"])
        if prev_latlon is None or cur_latlon is None:
            continue
        dt = (row["timestamp"] - prev_row["timestamp"]).total_seconds()
        dist_km = haversine(prev_latlon, cur_latlon, unit=Unit.KILOMETERS)
        if dist_km > 0 and 10 < dt < 1200:
            training_records.append({
                "route": row["route"],
                "direction": row["direction"],
                "from_stop": prev_row["staCode"],
                "to_stop": row["staCode"],
                "trip_time_s": dt,
                "distance_km": round(dist_km, 4),
                "hour": prev_row["timestamp"].hour,
                "day_of_week": prev_row["timestamp"].weekday(),
                "timestamp": prev_row["timestamp"].isoformat(),
            })
        prev_row = row
        prev_latlon = cur_latlon

train_df = pd.DataFrame(training_records)
print(f"Training records: {len(train_df)}")
print(f"Unique segments: {train_df.groupby(['from_stop','to_stop']).ngroups}")
print(f"Routes: {train_df['route'].value_counts().to_dict()}")
train_df.head()

In [ ]:
# Feature: is_holiday
HOLIDAYS_2026 = [
    "2026-01-01", "2026-01-29", "2026-01-30", "2026-01-31",
    "2026-04-04", "2026-04-05", "2026-05-01", "2026-05-06",
    "2026-06-19", "2026-09-22", "2026-10-01", "2026-10-02",
    "2026-10-29", "2026-12-08", "2026-12-20", "2026-12-21",
    "2026-12-24", "2026-12-25",
]
holiday_set = set(HOLIDAYS_2026)

# Feature: zone_type (based on stop code prefix)
def assign_zone(stop_code):
    code = str(stop_code)
    prefix = code.split("/")[0]
    if prefix in ["M1","M2","M3"]:
        return "old_town"
    elif prefix in ["M4","M6"]:
        return "bridge"
    elif prefix in ["M7","M8"]:
        return "cotai"
    elif code.startswith("T"):
        return "taipa"
    return "other"

train_df["is_holiday"] = train_df["timestamp"].apply(
    lambda t: t[:10] in holiday_set
).astype(int)
train_df["zone_type"] = train_df["from_stop"].apply(assign_zone)

print(f"is_holiday balance: {train_df['is_holiday'].value_counts().to_dict()}")
print(f"Zone distribution: {train_df['zone_type'].value_counts().to_dict()}")

In [ ]:
# Export clean CSV for training
output_cols = [
    "route","direction","from_stop","to_stop",
    "trip_time_s","distance_km","hour","day_of_week","is_holiday","zone_type"
]
train_df[output_cols].to_csv("training_data.csv", index=False)
print(f"Exported {len(train_df)} records to training_data.csv")
from google.colab import files
# files.download("training_data.csv")  # uncomment when ready

In [ ]:
# Quick EDA
import matplotlib.pyplot as plt

print("=== Trip time distribution ===")
print(train_df["trip_time_s"].describe())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
train_df["trip_time_s"].hist(bins=50, ax=axes[0])
axes[0].set_title("Trip time distribution (s)")
train_df.groupby("hour")["trip_time_s"].mean().plot(ax=axes[1])
axes[1].set_title("Mean trip time by hour")
train_df.groupby("zone_type")["trip_time_s"].mean().plot(kind="bar", ax=axes[2])
axes[2].set_title("Mean trip time by zone")
plt.tight_layout()
plt.show()